In [1]:
import leafmap

# 1. Crear el objeto mapa centrado en el Ecuador con zoom global
# Leafmap permite usar diversos mapas base por defecto
m = leafmap.Map(center=[0, 0], zoom=2)

# 2. Definir la ruta del archivo GeoJSON (Cables submarinos)
# Se utiliza una URL de release para mayor compatibilidad con los drivers GDAL
in_geojson = (
    "https://github.com/opengeos/datasets/releases/download/vector/cables.geojson"
)

# 3. Añadir la capa al mapa con un nombre específico
# Este método lee el archivo remoto y lo renderiza sobre el mapa base
m.add_geojson(in_geojson, layer_name="Líneas de Cable")

# 4. Mostrar el mapa interactivo
m

Map(center=[0, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text'…

In [1]:
import geemap

# 1. Crear el mapa centrado en un punto (Bogotá) para evitar errores de geometría
m = geemap.Map(center=[4.63, -74.08], zoom=10)

# 2. Cargar colección Sentinel-2 Armonizada (ID corArecto para 2024)
collection = (
    geemap.ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2024-01-01', '2024-03-31')
    .median()
)

# 3. Parámetros de visualización simples
vis_params = {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}

# 4. Añadir capa y mostrar
m.addLayer(collection, vis_params, 'Sentinel-2 UNAL')
m

Enter verification code:  4/1AfrIepDSzqo8cNzYoCC9ca1FwEzqUzjx0x4caVdc3lzAXqLAVrwX5UGqJQo



Successfully saved authorization token.


Map(center=[4.63, -74.08], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [3]:
import ee
import geemap

# Inicializar Earth Engine
ee.Initialize()

# Crear mapa centrado en Cundinamarca
Map = geemap.Map(center=[4.7, -74.1], zoom=8)

# ---------------------------
# 1️⃣ Cargar límites administrativos de Colombia
# ---------------------------
colombia = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Filtrar solo Cundinamarca
cundinamarca = colombia.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Colombia"),
        ee.Filter.eq("ADM1_NAME", "Cundinamarca")
    )
)

# ---------------------------
# 2️⃣ Cargar DEM SRTM
# ---------------------------
dem = ee.Image("USGS/SRTMGL1_003")

# Recortar al departamento
dem_clip = dem.clip(cundinamarca)

# ---------------------------
# 3️⃣ Generar Hillshade
# ---------------------------
hillshade = ee.Terrain.hillshade(dem_clip)

# ---------------------------
# 4️⃣ Parámetros de visualización
# ---------------------------
vis_params = {
    "min": 0,
    "max": 255,
    "palette": ["000000", "FFFFFF"]
}

# ---------------------------
# 5️⃣ Agregar al mapa
# ---------------------------
Map.addLayer(hillshade, vis_params, "Hillshade Cundinamarca")

# Añadir límite del departamento
Map.addLayer(cundinamarca, {}, "Límite Cundinamarca")

Map


Map(center=[4.7, -74.1], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [4]:
import ee
import geemap

# Inicializar Earth Engine
ee.Initialize()

# Crear mapa centrado en Cundinamarca
Map = geemap.Map(center=[4.7, -74.1], zoom=8)

# ---------------------------
# 1️⃣ Cargar límites administrativos de Colombia
# ---------------------------
colombia = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Filtrar solo Cundinamarca
cundinamarca = colombia.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Colombia"),
        ee.Filter.eq("ADM1_NAME", "Cundinamarca")
    )
)

# ---------------------------
# 2️⃣ Cargar DEM SRTM
# ---------------------------
dem = ee.Image("USGS/SRTMGL1_003")

# Recortar al departamento
dem_clip = dem.clip(cundinamarca)

# ---------------------------
# 3️⃣ Generar Hillshade
# ---------------------------
hillshade = ee.Terrain.hillshade(dem_clip)

# ---------------------------
# 4️⃣ Parámetros de visualización
# ---------------------------
vis_params = {
    "min": 0,
    "max": 255,
    "palette": ["000000", "FFFFFF"]
}

# ---------------------------
# 5️⃣ Agregar al mapa
# ---------------------------
Map.addLayer(hillshade, vis_params, "Hillshade Cundinamarca")

# Añadir límite del departamento
Map.addLayer(cundinamarca, {}, "Límite Cundinamarca")

Map


Map(center=[4.7, -74.1], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [2]:
import ee
ee.Authenticate()

Enter verification code:  4/1AfrIepAF1-aMKroQMvfdIFY_0xVkRkrX7jjrL6psdopK_fIos8zTv24Y-cQ



Successfully saved authorization token.


In [7]:
import ee
import geemap

# Inicializar Earth Engine
# ee.Authenticate()  # Solo la primera vez
ee.Initialize()

# Crear mapa
Map = geemap.Map(center=[4.7, -74.1], zoom=8)

# ---------------------------
# 1️⃣ Límite Cundinamarca
# ---------------------------
colombia = ee.FeatureCollection("FAO/GAUL/2015/level1")

cundinamarca = colombia.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Colombia"),
        ee.Filter.eq("ADM1_NAME", "Cundinamarca")
    )
)

# ---------------------------
# 2️⃣ DEM y pendiente (%)
# ---------------------------
dem = ee.Image("USGS/SRTMGL1_003").clip(cundinamarca)

slope_deg = ee.Terrain.slope(dem)

# Conversión correcta a porcentaje
slope_pct = slope_deg.multiply(3.14159).divide(180).tan().multiply(100)

# ---------------------------
# 3️⃣ Clasificación IGAC
# ---------------------------
clasificacion = (
    slope_pct
    .where(slope_pct.lte(3), 1)
    .where(slope_pct.gt(3).And(slope_pct.lte(7)), 2)
    .where(slope_pct.gt(7).And(slope_pct.lte(12)), 3)
    .where(slope_pct.gt(12).And(slope_pct.lte(25)), 4)
    .where(slope_pct.gt(25).And(slope_pct.lte(50)), 5)
    .where(slope_pct.gt(50).And(slope_pct.lte(75)), 6)
    .where(slope_pct.gt(75), 7)
)

# ---------------------------
# 4️⃣ Visualización
# ---------------------------
slope_vis = {
    "min": 1,
    "max": 7,
    "palette": [
        "#1a9641",  # 0–3
        "#a6d96a",  # 3–7
        "#ffffbf",  # 7–12
        "#fdae61",  # 12–25
        "#f46d43",  # 25–50
        "#d73027",  # 50–75
        "#7f0000"   # >75
    ]
}

Map.addLayer(clasificacion, slope_vis, "Pendiente IGAC (%)")
Map.addLayer(cundinamarca, {}, "Límite Cundinamarca")

# ---------------------------
# 5️⃣ Leyenda
# ---------------------------
legend_dict = {
    "0–3 % (Plano)": "#1a9641",
    "3–7 % (Ligeramente Inclinado)": "#a6d96a",
    "7–12 % (Moderadamente Inclinado)": "#ffffbf",
    "12–25 % (Fuertemente Inclinado)": "#fdae61",
    "25–50 % (Ligeramente Escarpado)": "#f46d43",
    "50–75 % (Moderadamente Escarpado)": "#d73027",
    ">75 % (Fuertemente Escarpado)": "#7f0000"
}

Map.add_legend(title="Clasificación Pendiente IGAC (%)",
               legend_dict=legend_dict)

# ---------------------------
# 6️⃣ Click para mostrar pendiente real
# ---------------------------
Map.addLayer(slope_pct, {"min": 0, "max": 100}, "Pendiente real (%)", False)

Map


Map(center=[4.7, -74.1], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …